In [ ]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import mysql.connector

# Connect to MySQL
conn_mysql = mysql.connector.connect(
    host="localhost",
    port=3305,
    user="root",
    password="mysql",
    database="Air_TrackerFlight_DB"
  )

# Create cursor
cursor_mysql = conn_mysql.cursor()

print("MySQL connection Established")

MySQL connection Established


In [12]:
cursor_mysql.execute("Create Database IF NOT EXISTS Air_TrackerFlight_DB")
print("Mysql database 'Air_TrackerFlight_DB' create sucessfully")



Mysql database 'Air_TrackerFlight_DB' create sucessfully


In [13]:
cursor_mysql.execute("USE Air_TrackerFlight_DB")
print("Mysql Use  air_TrackerFlight_DB sucessfully")


Mysql Use  air_TrackerFlight_DB sucessfully


In [14]:
#=========================Table Airport_data====================
cursor_mysql.execute("""
CREATE TABLE IF NOT EXISTS airport_data(
 airport_id INT PRIMARY KEY AUTO_INCREMENT,
    icao_code VARCHAR(10) UNIQUE,
    iata_code VARCHAR(10) UNIQUE,
    name VARCHAR(100),
    city VARCHAR(100),
    country VARCHAR(100),
    continent VARCHAR(50),
    latitude FLOAT,
    longitude FLOAT,
    timezone VARCHAR(50)
);
""")

conn_mysql.commit()

print("Table 'airport_data' created successfully in MySQL")

Table 'airport_data' created successfully in MySQL


In [15]:
cursor_mysql.execute("SHOW TABLES")
for table in cursor_mysql:
    print(table)

('aircraft_data',)
('airport_data',)
('airport_delays_data',)
('flights_data',)


In [16]:
import mysql.connector
import csv

# Create cursor
cursor_mysql = conn_mysql.cursor()

# Open CSV file
with open("clean_airport_data.csv", "r", encoding="utf-8") as file:

    csv_reader = csv.reader(file)

    next(csv_reader)   # Skip header row

    for row in csv_reader:

        query = """
INSERT INTO airport_data
(icao_code, iata_code, name, city, country, continent, latitude, longitude, timezone)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
    iata_code = VALUES(iata_code),
    name = VALUES(name),
    city = VALUES(city),
    country = VALUES(country),
    continent = VALUES(continent),
    latitude = VALUES(latitude),
    longitude = VALUES(longitude),
    timezone = VALUES(timezone)
"""

        cursor_mysql.execute(query, row)

# Save data
conn_mysql.commit()

print("CSV data inserted successfully")

cursor_mysql.close()
cursor_mysql.close()

CSV data inserted successfully


False

In [17]:
# Create cursor
cursor_mysql = conn_mysql.cursor()

# Create table
cursor_mysql.execute("""
CREATE TABLE IF NOT EXISTS flights_data(
    flight_id INT PRIMARY KEY AUTO_INCREMENT,
    flight_number VARCHAR(20),
    aircraft_registration VARCHAR(20),
    origin_iata VARCHAR(10),
    destination_iata VARCHAR(10),
    scheduled_departure DATETIME,
    actual_departure DATETIME,
    scheduled_arrival DATETIME,
    actual_arrival DATETIME,
    status VARCHAR(20),
    airline_code VARCHAR(10)
);
""")

conn_mysql.commit()
print("Table 'flights_data' created successfully")


Table 'flights_data' created successfully


In [19]:
import csv

def fix_date(dt):
    if dt and dt.strip() and dt != "NA":
        dt = dt.split("+")[0]   # remove timezone
        if len(dt) == 16:
            dt += ":00"
        return dt
    return None

query = """
INSERT INTO flights_data
(flight_number, aircraft_registration, origin_iata, destination_iata,
 scheduled_departure, actual_departure,
 scheduled_arrival, actual_arrival, status, airline_code)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

inserted = 0
skipped = 0

with open("cleaned_flight_data.csv", "r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)
    next(csv_reader)

    for row in csv_reader:
        try:
            # Remove flight_id column
            if len(row) == 11:
                row = row[1:]

            if len(row) != 10:
                print("Skipped:", row)
                skipped += 1
                continue

            # Fix datetime columns
            row[4] = fix_date(row[4])
            row[5] = fix_date(row[5])
            row[6] = fix_date(row[6])
            row[7] = fix_date(row[7])

            # Convert NA / empty → None
            row = [None if x in ("", "NA", "Unknown") else x for x in row]

            cursor_mysql.execute(query, row)
            inserted += 1

        except Exception as e:
            print("ERROR:", e)
            print("ROW:", row)
            skipped += 1

conn_mysql.commit()

print("✅ Inserted:", inserted)
print("⚠️ Skipped:", skipped)

ERROR: 1292 (22007): Incorrect datetime value: '2025-04-05 05:45-04:00' for column 'scheduled_arrival' at row 1
ROW: ['AC 43', 'C-FIUF', 'DEL', 'YYZ', '2025-04-04 22:50:00', '2025-04-04 22:50:00', '2025-04-05 05:45-04:00', '2025-04-05 05:37-04:00', 'Departed', 'AC']
ERROR: 1292 (22007): Incorrect datetime value: '2025-04-05 06:05-04:00' for column 'scheduled_arrival' at row 1
ROW: ['AA 293', 'N839AA', 'DEL', 'JFK', '2025-04-04 23:30:00', '2025-04-04 23:30:00', '2025-04-05 06:05-04:00', '2025-04-05 06:04-04:00', 'Departed', 'AA']
ERROR: 1292 (22007): Incorrect datetime value: '2025-04-05 06:25-04:00' for column 'scheduled_arrival' at row 1
ROW: ['UA 83', 'N29971', 'DEL', 'EWR', '2025-04-04 23:35:00', '2025-04-04 23:35:00', '2025-04-05 06:25-04:00', '2025-04-05 06:43-04:00', 'Departed', 'UA']
ERROR: 1292 (22007): Incorrect datetime value: '2025-04-05 06:25-04:00' for column 'scheduled_arrival' at row 1
ROW: ['AC 51', 'C-FVLQ', 'DEL', 'YUL', '2025-04-05 00:05:00', '2025-04-05 00:05:00', '

In [40]:
#========================= Table Aircrft_data ====================
cursor_mysql = conn_mysql.cursor().execute("""
CREATE TABLE IF NOT EXISTS aircraft_data(
    aircraft_id INT PRIMARY KEY AUTO_INCREMENT,
    registration VARCHAR(20) UNIQUE,
    model VARCHAR(100),
    manufacturer VARCHAR(100),
    icao_type_code VARCHAR(20),
    owner VARCHAR(100)
);
""")

conn_mysql.commit()

print("Table 'aircraft_data' created successfully in MySQL")

Table 'aircraft_data' created successfully in MySQL


In [55]:

cursor_mysql.execute("SHOW TABLES")
for table in cursor_mysql:
    print(table)



('aircraft_data',)
('airport_data',)
('flights_data',)


In [48]:
import mysql.connector
import csv

# Create cursor
cursor_mysql = conn_mysql.cursor()

# Open CSV file
with open("aircrafts_data.csv", "r", encoding="utf-8") as file:

    csv_reader = csv.reader(file)

    next(csv_reader)   # Skip header row

    for row in csv_reader:

        query = """
        INSERT INTO aircraft_data
        ( aircraft_id,registration,model, manufacturer,icao_type_code,owner)
        VALUES (%s,%s,%s,%s,%s,%s)
        """

        cursor_mysql.execute(query, row)

# Save data
conn_mysql.commit()

print("CSV data inserted successfully")

cursor_mysql.close()
cursor_mysql.close()

CSV data inserted successfully


False

In [57]:
#========================= airport_delays_data ====================
cursor_mysql = conn_mysql.cursor()

cursor_mysql.execute("""
CREATE TABLE IF NOT EXISTS airport_delays_data(
    delay_id INT PRIMARY KEY AUTO_INCREMENT,
    airport_iata VARCHAR(20),
    delay_date DATETIME,
    total_flights INT,
    delayed_flights INT,
    avg_delay_min FLOAT,
    canceled_flights INT
);
""")

conn_mysql.commit()

print("Table 'airport_delays_data' created successfully in MySQL")

Table 'airport_delays_data' created successfully in MySQL


In [59]:
import csv
from datetime import datetime

cursor_mysql = conn_mysql.cursor()

query = """
INSERT INTO airport_delays_data
(airport_iata, delay_date, total_flights,
 delayed_flights, avg_delay_min, canceled_flights)
VALUES (%s,%s,%s,%s,%s,%s)
"""

def fix_date(dt):
    if dt:
        return datetime.strptime(dt, "%d-%m-%Y").strftime("%Y-%m-%d %H:%M:%S")
    return None

inserted = 0
skipped = 0

with open("airport_delay_data.csv", "r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)
    next(csv_reader)

    for row in csv_reader:
        try:
            print("RAW:", row)

            # ✅ Your CSV has 8 columns → fix mapping
            if len(row) == 8:
                # Remove delay_id and extra column
                row = [
                    row[1],  # airport_iata
                    fix_date(row[2]),  # date
                    row[3],  # total_flights
                    row[4],  # delayed_flights
                    row[5],  # avg_delay_min
                    row[7]   # canceled_flights
                ]
            else:
                print("Skipped:", row)
                skipped += 1
                continue

            cursor_mysql.execute(query, row)
            inserted += 1

        except Exception as e:
            print("ERROR:", e)
            print("ROW:", row)
            skipped += 1

conn_mysql.commit()

print("✅ Inserted:", inserted)
print("⚠️ Skipped:", skipped)

cursor_mysql.close()

RAW: ['1', 'DEL', '04-04-2025', '752', '119', '59', '39', '6']
RAW: ['2', 'BOM', '04-04-2025', '493', '37', '40', '35', '2']
RAW: ['3', 'BLR', '04-04-2025', '513', '82', '41', '25', '9']
RAW: ['4', 'MAA', '04-04-2025', '313', '10', '22', '8', '0']
RAW: ['5', 'HYD', '04-04-2025', '343', '14', '19', '15', '1']
RAW: ['6', 'CCU', '04-04-2025', '227', '13', '22', '30', '0']
RAW: ['7', 'AMD', '04-04-2025', '198', '33', '20', '13', '0']
RAW: ['8', 'COK', '04-04-2025', '142', '8', '15', '14', '0']
RAW: ['9', 'GOI', '04-04-2025', '79', '1', '10', '10', '0']
RAW: ['10', 'PNQ', '04-04-2025', '179', '5', '11', '10', '1']
RAW: ['11', 'LKO', '04-04-2025', '110', '4', '9', '10', '1']
RAW: ['12', 'PAT', '04-04-2025', '44', '1', '10', '10', '0']
RAW: ['13', 'IXC', '04-04-2025', '49', '4', '18', '18', '0']
RAW: ['14', 'TRV', '04-04-2025', '73', '4', '30', '21', '2']
RAW: ['15', 'IXM', '04-04-2025', '52', '4', '25', '25', '0']
✅ Inserted: 15
⚠️ Skipped: 0


True